# Information Horizon on MELD — Qwen2.5-VL-3B

Measure **I_d(k) = P(y | only visual token k retained at depth d) − P(y | no visual tokens retained at depth d)**.
Both branches preserve all text/special tokens. Depth means completed decoder blocks; depth 0 is before the decoder.

**Before running:** enable a Kaggle GPU and Internet; attach your updated **benchmark-code** dataset and **meld-dataset**.
The code dataset must contain the reorganized repository and Information Horizon scripts. This notebook does not fetch an older GitHub checkout.

Default readout: the **fixed full-baseline emotion MLP**, with its saved normalization. This adapts the paper's intervention to our benchmark.
Set `READOUT = "lm"` for raw first-answer-token probabilities from the LM vocabulary softmax, as in the paper.
Neither mode trains any model. The LM's seven extracted probabilities need not sum to one.

[Paper, Section 3.2](https://arxiv.org/html/2512.07580v2) · [Official code](https://github.com/YahongWang1/Information-Horizon)


## 1. Paths and configuration
Use the same video/prompt/pooling settings as the baseline. The pilot scores 8 evenly spaced visual tokens at 4 depths in one dev sample.
It is a plumbing check, not an exhaustive token analysis. `RUN_SWEEP` is off initially. After inspecting the pilot, enable it and rerun from the scoring cells.
The default sweep scores every token and every decoder depth for 10 samples; start smaller if necessary.


In [ ]:
from pathlib import Path

CODE = Path("/kaggle/input/datasets/pushkarsingh2005/benchmark-code")
MELD_ROOT = Path("/kaggle/input/datasets/pushkarsingh2005/meld-dataset/MELD.Raw")
READOUT = "mlp"                     # "mlp" or "lm"
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
CHECKPOINT_OVERRIDE = None           # Optional explicit Path to full-baseline best_mlp.pt
SPLIT = "dev"                       # Use dev for exploratory analysis
START = 0                           # Row offset within the rebuilt split index
FPS, MIN_FRAMES, MAX_FRAMES = 6, 4, 64
FRAME_SIZE = 224
POOLING, PROMPT_STYLE, DTYPE = "last", "emotion_task", "float16"
PILOT_DEPTHS, PILOT_TOKENS = "0,8,16,24", 8
RUN_SWEEP = False
SWEEP_DEPTHS = "all"
SWEEP_TOKENS = 0                     # 0 = exhaustive; >0 = evenly spaced subset
SWEEP_SAMPLES = 10
RUN_NAME = f"qwen2_5_vl_3b_{SPLIT}_{READOUT}_horizon_v1"
RUN_ROOT = Path("/kaggle/working/experiments/information_horizon") / RUN_NAME
RESTORE_FROM = None                  # Optional Path to previous RUN_ROOT in attached Kaggle output

INDEX_ROOT = RUN_ROOT / "data"
INDEX_CSV = INDEX_ROOT / f"meld_{SPLIT}_index.csv"
PILOT_DIR = RUN_ROOT / "pilot"
SWEEP_DIR = RUN_ROOT / "sweep"
RUNNER = CODE / "scripts/information_horizon/run_information_horizon.py"
SUMMARIZER = CODE / "scripts/information_horizon/summarize_information_horizon.py"
assert READOUT in {"mlp", "lm"} and SPLIT in {"dev", "test", "train"}
assert START >= 0 and SWEEP_SAMPLES > 0 and SWEEP_TOKENS >= 0
for required in [MELD_ROOT, RUNNER, SUMMARIZER, CODE / "scripts/__init__.py",
                 CODE / "tests/test_information_horizon.py", CODE / "requirements.txt"]:
    assert required.exists(), f"Missing {required}. Upload the updated reorganized benchmark-code dataset."
CHECKPOINT = None
if READOUT == "mlp":
    candidates = ([Path(CHECKPOINT_OVERRIDE)] if CHECKPOINT_OVERRIDE is not None else [
        CODE / "experiments/baselines/qwen2_5_vl_3b_shared/outputs/best_mlp.pt",
        CODE / "outputs/qwen2_5_vl_3b_shared/best_mlp.pt",  # Legacy checkpoint-only fallback
    ])
    CHECKPOINT = next((p for p in candidates if p.is_file()), None)
    assert CHECKPOINT is not None, "Full-baseline MLP missing: set CHECKPOINT_OVERRIDE. Do not use a pruned-run MLP."
print("Code:", CODE, "\nReadout:", READOUT, "\nCheckpoint:", CHECKPOINT, "\nOutput:", RUN_ROOT)


## 2. Install dependencies
Keep Kaggle's CUDA-enabled PyTorch. Pin Transformers **4.57.1**, the version used by the decoder-hook integration test; do not reuse the Transformers 5.0 override from other notebooks.
Run this before importing Transformers. If you already imported another version, restart the session after installation.


In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.1", "qwen-vl-utils[decord]==0.0.14",
    "accelerate", "decord", "pandas", "numpy", "scikit-learn",
    "matplotlib", "Pillow", "tqdm", "PyYAML", "opencv-python-headless",
], check=True)


In [ ]:
import csv
import hashlib
import json
import os
import shutil
import time
import numpy as np
import pandas as pd
import torch
import transformers
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

assert transformers.__version__ == "4.57.1", "Restart the kernel to load the pinned Transformers version."
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator."
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}, {props.total_memory / 2**30:.1f} GiB")
print("Torch:", torch.__version__, "Transformers:", transformers.__version__)
# The runner uses eager attention, never FlashAttention, and disables the KV cache.
os.environ["FORCE_QWENVL_VIDEO_READER"] = "decord"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
if CHECKPOINT is not None:
    saved = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
    assert saved["input_dim"] == 2048, "Expected the Qwen2.5-VL-3B baseline MLP."
    assert saved["label_names"] == ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
    print("Baseline checkpoint SHA256:", hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest())
    print("Checkpoint training args:", saved.get("args", {}))
    del saved
# Small randomly initialized model: validates hooks without downloading Qwen weights.
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(CODE / "tests"),
                "-p", "test_information_horizon.py", "-v"], cwd=CODE, check=True)


## 3. Restore an earlier run (optional), then rebuild the index
Kaggle input files are read-only. Outputs stay under `/kaggle/working`.
To continue a saved run, attach its output, set `RESTORE_FROM` to the extracted run directory (containing `pilot/` or `sweep/`), and keep the same settings and software versions.
Completed samples are reused; an interrupted sample is recomputed. The runner rejects changed manifests. No partial-sample resume is promised.


In [ ]:
RUN_ROOT.mkdir(parents=True, exist_ok=True)
if RESTORE_FROM is not None:
    source = Path(RESTORE_FROM)
    assert source.is_dir() and source.resolve() != RUN_ROOT.resolve()
    assert (source / "pilot/config.json").exists() or (source / "sweep/config.json").exists(), "Select the previous run directory, not its parent."
    for src in source.rglob("*"):
        if not src.is_file():
            continue
        dst = RUN_ROOT / src.relative_to(source)
        if dst.exists():
            assert hashlib.sha256(dst.read_bytes()).digest() == hashlib.sha256(src.read_bytes()).digest(), f"Conflicting restore file: {dst}"
        else:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
    print("Restored:", source)

# Preserve provenance separately: scoring folders must contain only runner JSON files.
source_hashes = {str(p.relative_to(CODE)): hashlib.sha256(p.read_bytes()).hexdigest()
                 for folder in ["scripts/information_horizon", "scripts/extraction", "scripts/classification", "scripts/pruning"]
                 for p in (CODE / folder).glob("*.py")}
provenance = RUN_ROOT / "source_hashes.json"
if provenance.exists():
    assert json.loads(provenance.read_text()) == source_hashes, "Source changed: use a fresh RUN_NAME."
else:
    provenance.write_text(json.dumps(source_hashes, indent=2))

subprocess.run([sys.executable, str(CODE / "scripts/data/prepare_meld.py"),
                "--meld-root", str(MELD_ROOT), "--split", SPLIT,
                "--output-dir", str(INDEX_ROOT)], cwd=CODE, check=True)
index = pd.read_csv(INDEX_CSV)
assert not index["sample_id"].duplicated().any()
assert START < len(index)
assert index["video_path"].map(lambda p: Path(p).is_file()).all()
if RUN_SWEEP:
    assert START + SWEEP_SAMPLES <= len(index), "Sweep exceeds available samples."
display(index.iloc[START:START + min(SWEEP_SAMPLES, 10)][["sample_id", "emotion", "utterance", "video_path"]])
print("Available samples:", len(index))


## 4. Command builder and logging
Each process loads the model once. For N scored visual tokens and D depths, a sample requires **1 + D × (N + 1)** forward passes.
Vision and prefix features are currently recomputed for every intervention. Eager attention at the full 64-frame budget can be memory-intensive.
Do not silently reduce sampling settings and compare against an unmatched baseline; use a separate run name if you change them.


In [ ]:
def build_command(output_dir, depths, tokens, samples):
    cmd = [sys.executable, "-u", str(RUNNER),
           "--index-csv", str(INDEX_CSV), "--output-dir", str(output_dir),
           "--model-id", MODEL_ID, "--readout", READOUT,
           "--depths", str(depths), "--max-tokens", str(tokens), "--limit", str(samples),
           "--start", str(START), "--fps", str(FPS), "--min-frames", str(MIN_FRAMES),
           "--max-frames", str(MAX_FRAMES), "--frame-size", str(FRAME_SIZE),
           "--pooling", POOLING, "--prompt-style", PROMPT_STYLE, "--dtype", DTYPE, "--resume"]
    if READOUT == "mlp":
        cmd += ["--checkpoint", str(CHECKPOINT)]
    return cmd

def run_logged(cmd, name):
    import shlex
    log_dir = RUN_ROOT / "logs"
    log_dir.mkdir(exist_ok=True)
    print(shlex.join(cmd), flush=True)
    started = time.monotonic()
    with (log_dir / f"{name}.log").open("a") as log:
        log.write("\nCOMMAND: " + shlex.join(cmd) + "\n")
        with subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                status = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if status:
        raise subprocess.CalledProcessError(status, cmd)
    print(f"Finished in {(time.monotonic()-started)/60:.1f} minutes")

def load_samples(folder):
    return [json.loads(p.read_text()) for p in sorted(Path(folder).glob("*.json")) if p.name != "config.json"]


## 5. One-sample pilot
The first run downloads Qwen weights. Progress is printed per sample and depth; one depth can take a while. This pilot recomputes its own full-token baseline.


In [ ]:
run_logged(build_command(PILOT_DIR, PILOT_DEPTHS, PILOT_TOKENS, 1), "pilot")
pilot_samples = load_samples(PILOT_DIR)
assert len(pilot_samples) == 1
pilot = pilot_samples[0]
for layer in pilot["depths"]:
    absent = np.asarray(layer["absent_probabilities"])
    for item in layer["tokens"]:
        present = np.asarray(item["probabilities"])
        delta = np.asarray(item["probability_differences"])
        assert np.isfinite(present).all() and np.isfinite(absent).all()
        assert np.allclose(delta, present - absent, atol=1e-6)
        y = pilot["labels"].index(pilot["true_label"])
        assert np.isclose(item["information_score"], delta[y], atol=1e-6)
        if READOUT == "mlp":
            assert np.isclose(present.sum(), 1, atol=1e-5) and np.isclose(absent.sum(), 1, atol=1e-5)
print("Pilot validated:", pilot["sample_id"], "ground truth:", pilot["true_label"])
display(pd.DataFrame({"emotion": pilot["labels"], "full_probability": pilot["full_probabilities"]}))
from transformers import AutoConfig
model_config = AutoConfig.from_pretrained(MODEL_ID)
text_config = getattr(model_config, "text_config", model_config)
sweep_depth_count = text_config.num_hidden_layers if SWEEP_DEPTHS == "all" else len(set(map(int, SWEEP_DEPTHS.split(','))))
sweep_tokens = pilot["visual_token_count"] if SWEEP_TOKENS == 0 else min(SWEEP_TOKENS, pilot["visual_token_count"])
print("Approximate sweep forward-pass count using pilot length:", SWEEP_SAMPLES * (1 + sweep_depth_count * (sweep_tokens + 1)))
print("Actual video lengths/token counts vary; this is not a runtime estimate.")


## 6. Optional sweep
Enable `RUN_SWEEP` in the configuration cell after reviewing the pilot. Keep `SWEEP_TOKENS=0` for exhaustive token scoring.
A positive cap produces sampled-token statistics and must be reported as such. This uses the first selected contiguous index rows, not a representative random sample.
Change `RUN_NAME` for changes to the scoring configuration. Do not interpret a ten-sample pilot as a dataset-level conclusion.


In [ ]:
if RUN_SWEEP:
    run_logged(build_command(SWEEP_DIR, SWEEP_DEPTHS, SWEEP_TOKENS, SWEEP_SAMPLES), "sweep")
else:
    print("Sweep disabled. Set RUN_SWEEP=True in the configuration cell to launch it.")
ANALYSIS_DIR = SWEEP_DIR if RUN_SWEEP else PILOT_DIR
ANALYSIS_LABEL = "sweep" if RUN_SWEEP else "pilot"
subprocess.run([sys.executable, str(SUMMARIZER), "--input-dir", str(ANALYSIS_DIR)], cwd=CODE, check=True)
summary = pd.read_csv(ANALYSIS_DIR / "layer_summary.csv")
display(summary)


## 7. Export detailed probability tables
Positive score means the retained token increases the correct-class probability relative to retaining no visual tokens **at that same depth**.
Negative scores remain negative. Full-minus-absent measures collective visual-token contribution and is a different quantity.


In [ ]:
samples = load_samples(ANALYSIS_DIR)
assert samples, "No completed samples to analyze."
assert len(samples) == (SWEEP_SAMPLES if RUN_SWEEP else 1), "Incomplete run: finish scoring before analysis."
REPORT_DIR = RUN_ROOT / "analysis" / ANALYSIS_LABEL
REPORT_DIR.mkdir(parents=True, exist_ok=True)
probability_rows, token_rows = [], []
for sample in samples:
    labels = sample["labels"]
    y = labels.index(sample["true_label"])
    for layer in sample["depths"]:
        for j, label in enumerate(labels):
            probability_rows.append(dict(sample_id=sample["sample_id"], true_label=sample["true_label"],
                depth=layer["depth"], emotion=label, full_probability=sample["full_probabilities"][j],
                absent_probability=layer["absent_probabilities"][j],
                full_minus_absent=layer["full_minus_absent"][j]))
        for item in layer["tokens"]:
            record = dict(sample_id=sample["sample_id"], true_label=sample["true_label"], depth=layer["depth"],
                          token_ordinal=item["token_ordinal"], sequence_position=item["sequence_position"],
                          information_score=item["information_score"])
            for j, label in enumerate(labels):
                record[f"present_prob_{label}"] = item["probabilities"][j]
                record[f"absent_prob_{label}"] = layer["absent_probabilities"][j]
                record[f"delta_{label}"] = item["probability_differences"][j]
            token_rows.append(record)
probabilities = pd.DataFrame(probability_rows)
tokens = pd.DataFrame(token_rows)
probabilities.to_csv(REPORT_DIR / "full_vs_absent_probabilities.csv", index=False)
tokens.to_csv(REPORT_DIR / "token_information_scores.csv", index=False)
summary.to_csv(REPORT_DIR / "layer_summary.csv", index=False)
display(tokens.head())


## 8. Plot information versus depth
Signed means can cancel; inspect absolute magnitude, variance, and prediction changes together.
In LM mode, correctness below is restricted to the seven emotion first-token scores, not generated-answer accuracy.
These plots do not automatically declare a horizon.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes[0, 0].plot(summary.depth, summary.mean_information, "o-", label="Mean signed I")
axes[0, 0].plot(summary.depth, summary.mean_absolute_information, "o-", label="Mean |I|")
axes[0, 0].axhline(0, color="gray", linewidth=.7)
axes[0, 0].set_ylabel("Probability difference")
axes[0, 0].legend()
axes[0, 1].plot(summary.depth, summary.within_sample_variance, "o-")
axes[0, 1].set_ylabel("Mean within-sample variance of I")
axes[1, 0].plot(summary.depth, summary.full_correct, "o-", label="Full tokens")
axes[1, 0].plot(summary.depth, summary.absent_correct, "o-", label="No visual tokens after depth")
axes[1, 0].set_ylabel("Accuracy on selected samples")
axes[1, 0].set_ylim(-.03, 1.03)
axes[1, 0].legend()
axes[1, 1].plot(summary.depth, summary.full_minus_absent_true, "o-")
axes[1, 1].axhline(0, color="gray", linewidth=.7)
axes[1, 1].set_ylabel("Mean P_full(y) − P_absent(y)")
for ax in axes.flat:
    ax.set_xlabel("Completed decoder blocks")
    ax.grid(alpha=.2)
fig.suptitle(f"{ANALYSIS_LABEL}: {READOUT}, n={len(samples)} (selected samples)")
fig.savefig(REPORT_DIR / "information_horizon.png", dpi=160)
plt.show()

# Ordinal heatmap, not a spatial facial-attention map.
example_id = samples[0]["sample_id"]
heat = tokens[tokens.sample_id == example_id].pivot(index="depth", columns="token_ordinal", values="information_score")
scale = max(float(np.abs(heat.to_numpy()).max()), 1e-8)
fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)
im = ax.imshow(heat.to_numpy(), aspect="auto", cmap="coolwarm", vmin=-scale, vmax=scale)
ax.set_yticks(range(len(heat.index)), labels=heat.index)
ticks = np.unique(np.linspace(0, len(heat.columns)-1, min(10, len(heat.columns))).astype(int))
ax.set_xticks(ticks, labels=heat.columns[ticks])
ax.set_xlabel("Visual token ordinal (sampled columns may have gaps)")
ax.set_ylabel("Completed decoder blocks")
ax.set_title(example_id + " — signed token information")
fig.colorbar(im, ax=ax, label="P_single(y) − P_absent(y)")
fig.savefig(REPORT_DIR / "token_information_heatmap.png", dpi=160)
plt.show()


## 9. Compare recomputed full probabilities to the saved baseline (MLP only)
This is a diagnostic, not the information score. Small differences can arise from saved-embedding precision or software/backend differences.
A large discrepancy requires investigating preprocessing, prompt, pooling, checkpoint, and precision before interpreting the intervention.


In [ ]:
if READOUT == "mlp":
    candidates = [CODE / f"experiments/baselines/qwen2_5_vl_3b_shared/outputs/{SPLIT}_predictions.csv",
                  CODE / f"outputs/qwen2_5_vl_3b_shared/{SPLIT}_predictions.csv"]
    baseline_path = next((p for p in candidates if p.is_file()), None)
    if baseline_path is None:
        print("Saved baseline CSV is unavailable; the recomputed full probabilities remain in sample JSON files.")
    else:
        baseline = pd.read_csv(baseline_path)
        assert not baseline.sample_id.duplicated().any()
        baseline = baseline.set_index("sample_id")
        comparison = []
        for s in samples:
            if s["sample_id"] not in baseline.index:
                continue
            row = baseline.loc[s["sample_id"]]
            assert row.true_label == s["true_label"]
            previous = row[[f"prob_{c}" for c in s["labels"]]].to_numpy(dtype=float)
            current = np.asarray(s["full_probabilities"])
            comparison.append(dict(sample_id=s["sample_id"], max_abs_probability_difference=float(np.abs(current-previous).max()),
                                   same_predicted_class=bool(current.argmax() == previous.argmax())))
        comparison = pd.DataFrame(comparison)
        comparison.to_csv(REPORT_DIR / "saved_baseline_comparison.csv", index=False)
        print(f"Matched {len(comparison)} of {len(samples)} analyzed samples to saved baseline.")
        display(comparison)
else:
    print("LM vocabulary probabilities are not comparable to saved MLP class probabilities.")


## 10. Download and resume
The bundle contains configs, source hashes, the rebuilt index, logs, completed sample JSON files, CSV tables, and plots.
It excludes videos, model weights, and the classifier checkpoint. Save a Kaggle notebook version to retain `/kaggle/working` output.
For a future session, attach/extract the saved run and configure `RESTORE_FROM` in cell 1.

Interpretation limits: the sampled subset is exploratory; text states at later depths already contain visual evidence.
Near-zero single-token scores do not alone prove collective visual irrelevance. Ground-truth-guided token rankings are oracle diagnostics, not an inference-time selector.


In [ ]:
# Store the zip outside RUN_ROOT to avoid recursively archiving the bundle itself.
archive_base = Path("/kaggle/working") / RUN_NAME
archive = shutil.make_archive(str(archive_base), "zip", root_dir=RUN_ROOT.parent, base_dir=RUN_ROOT.name)
print("Run directory:", RUN_ROOT)
print("Download bundle:", archive)
# Notebook file links are relative to the notebook server's working directory.
display(FileLink(os.path.relpath(archive, Path.cwd())))
